# World Bank GDP per Capita (PPP) — Full-Scale Re-Test of the Government Digital Policy Finding

**Source:** World Bank WDI, `NY.GDP.PCAP.PP.CD` (GDP per capita, PPP, current 
international $). Download: 
https://api.worldbank.org/v2/en/indicator/NY.GDP.PCAP.PP.CD?downloadformat=csv 
— unzip and place `API_NY.GDP.PCAP.PP.CD_DS2_en_csv_v2_*.csv` in `../data/raw/`.

**Purpose:** The Government Digital Policy finding — Nepal's headline result — 
was only ever income-adjusted using 10 hand-collected GDP figures. This rebuilds 
that test on the full ~195-country Oxford sample with real World Bank data,

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OXFORD_FILE = RAW_DIR / "2025GovernmentAIReadinessIndexdata.xlsx"

# Robust glob instead of a hardcoded filename — works regardless of dots vs underscores
gdp_candidates = [
    f for f in RAW_DIR.glob("*.csv")
    if "PCAP" in f.name.upper() and "PP" in f.name.upper() and "Metadata" not in f.name
]
meta_candidates = [
    f for f in RAW_DIR.glob("*.csv")
    if "Metadata_Country" in f.name and "PCAP" in f.name.upper()
]

if not gdp_candidates:
    raise FileNotFoundError(
        f"No GDP data file found in {RAW_DIR.resolve()}. Files present: "
        f"{[f.name for f in RAW_DIR.glob('*.csv')]}"
    )
if not meta_candidates:
    raise FileNotFoundError(
        f"No GDP metadata file found in {RAW_DIR.resolve()}. Files present: "
        f"{[f.name for f in RAW_DIR.glob('*.csv')]}"
    )

WB_GDP_FILE = gdp_candidates[0]
WB_META_FILE = meta_candidates[0]
print("GDP data file:  ", WB_GDP_FILE.name)
print("GDP metadata file:", WB_META_FILE.name)

GDP data file:   API_NY.GDP.PCAP.PP.CD_DS2_en_csv_v2_102948.csv
GDP metadata file: Metadata_Country_API_NY.GDP.PCAP.PP.CD_DS2_en_csv_v2_102948.csv


In [2]:
wb = pd.read_csv(WB_GDP_FILE, skiprows=4)
wb_meta = pd.read_csv(WB_META_FILE)

# Filter out aggregates (regions, income groups) — keep only real countries
real_country_codes = set(wb_meta.loc[wb_meta["Region"].notna(), "Country Code"])
wb = wb[wb["Country Code"].isin(real_country_codes)]

year_cols = [c for c in wb.columns if c.isdigit()]

def latest_value(row):
    for y in sorted(year_cols, reverse=True):
        if pd.notna(row[y]):
            return row[y], int(y)
    return np.nan, np.nan

wb[["gdp_pc_ppp", "gdp_year"]] = wb.apply(lambda r: pd.Series(latest_value(r)), axis=1)
wb_clean = wb[["Country Name", "Country Code", "gdp_pc_ppp", "gdp_year"]].dropna(subset=["gdp_pc_ppp"])
print(f"GDP data available for {len(wb_clean)} real countries (aggregates excluded).")
print(f"Year coverage: {wb_clean.gdp_year.min():.0f}–{wb_clean.gdp_year.max():.0f}")
wb_clean[wb_clean["Country Name"] == "Nepal"]

GDP data available for 203 real countries (aggregates excluded).
Year coverage: 2011–2025


,Country Name,Country Code,gdp_pc_ppp,gdp_year
177,Nepal,NPL,6175.158972,2025.0


In [3]:
xl = pd.ExcelFile(OXFORD_FILE)
oxford = xl.parse("Dimensions-Pillars", header=1)
oxford = oxford.drop(columns=[c for c in oxford.columns if "Unnamed" in str(c)])
oxford = oxford.dropna(subset=["Country"]).reset_index(drop=True)

NAME_ALIASES = {
    "United States of America": "United States",
    "Russia": "Russian Federation",
    "South Korea": "Korea, Rep.",
    "North Korea": "Korea, Dem. People's Rep.",
    "Egypt": "Egypt, Arab Rep.",
    "Iran": "Iran, Islamic Rep.",
    "Syria": "Syrian Arab Republic",
    "Venezuela": "Venezuela, RB",
    "Laos": "Lao PDR",
    "Democratic Republic of the Congo": "Congo, Dem. Rep.",
    "Republic of the Congo": "Congo, Rep.",
    "Ivory Coast": "Cote d'Ivoire",
    "Turkey": "Turkiye",
    "Slovakia": "Slovak Republic",
    "Vietnam": "Viet Nam",
    "Brunei": "Brunei Darussalam",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Gambia": "Gambia, The",
    "Bahamas": "Bahamas, The",
    "Yemen": "Yemen, Rep.",
    "Czech Republic": "Czechia",
    "Micronesia": "Micronesia, Fed. Sts.",
    "Saint Lucia": "St. Lucia",
    "Saint Kitts and Nevis": "St. Kitts and Nevis",
    "Saint Vincent and the Grenadines": "St. Vincent and the Grenadines",
}

oxford["match_name"] = oxford["Country"].replace(NAME_ALIASES)
merged = oxford.merge(wb_clean, left_on="match_name", right_on="Country Name", how="left")

unmatched = merged[merged["gdp_pc_ppp"].isna()]["Country"].tolist()
print(f"Matched: {merged['gdp_pc_ppp'].notna().sum()} of {len(merged)} Oxford countries")
print(f"Unmatched ({len(unmatched)}):")
print(unmatched)

assert merged.loc[merged.Country == "Nepal", "gdp_pc_ppp"].notna().iloc[0], "Nepal failed to match — stop and fix before proceeding"

Matched: 173 of 195 Oxford countries
Unmatched (22):
['Bolivia (Plurinational State of)', 'Congo', "Côte D'Ivoire", 'Cuba', "Democratic People's Republic of Korea", 'Gambia (Republic of The)', 'Guinea Bissau', 'Iran (Islamic Republic of)', "Lao People's Democratic Republic", 'Liechtenstein', 'Micronesia (Federated States of)', 'Monaco', 'Nauru', 'Republic of Korea', 'Republic of Moldova', 'Somalia', 'State of Palestine', 'Taiwan', 'Türkiye', 'United Kingdom of Great Britain and Northern Ireland', 'United Republic of Tanzania', 'Venezuela, Bolivarian Republic of']


In [4]:
NAME_ALIASES.update({
    "Bolivia (Plurinational State of)": "Bolivia",
    "Congo": "Congo, Rep.",
    "Côte D'Ivoire": "Cote d'Ivoire",
    "Gambia (Republic of The)": "Gambia, The",
    "Guinea Bissau": "Guinea-Bissau",
    "Iran (Islamic Republic of)": "Iran, Islamic Rep.",
    "Lao People's Democratic Republic": "Lao PDR",
    "Micronesia (Federated States of)": "Micronesia, Fed. Sts.",
    "Republic of Korea": "Korea, Rep.",
    "Republic of Moldova": "Moldova",
    "State of Palestine": "West Bank and Gaza",
    "Türkiye": "Turkiye",
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "United Republic of Tanzania": "Tanzania",
    "Venezuela, Bolivarian Republic of": "Venezuela, RB",
})

# Rerun the merge with the expanded alias set
oxford["match_name"] = oxford["Country"].replace(NAME_ALIASES)
merged = oxford.merge(wb_clean, left_on="match_name", right_on="Country Name", how="left")

unmatched = merged[merged["gdp_pc_ppp"].isna()]["Country"].tolist()
print(f"Matched: {merged['gdp_pc_ppp'].notna().sum()} of {len(merged)} Oxford countries")
print(f"Still unmatched ({len(unmatched)}):")
print(unmatched)

Matched: 188 of 195 Oxford countries
Still unmatched (7):
['Cuba', "Democratic People's Republic of Korea", 'Liechtenstein', 'Monaco', 'Nauru', 'Somalia', 'Taiwan']


In [5]:
merged["log_gdp"] = np.log(merged["gdp_pc_ppp"])
test_df = merged.dropna(subset=["log_gdp"])

TEST_COLS = ["Government digital policy", "Compute capacity", "Total Score", "Governance", "AI Infrastructure"]

print(f"Full-sample test, n={len(test_df)} countries\n")
for col in TEST_COLS:
    sub = test_df.dropna(subset=[col]).copy()
    slope, intercept, r, p, se = stats.linregress(sub["log_gdp"], sub[col])
    sub["resid"] = sub[col] - (slope * sub["log_gdp"] + intercept)

    nepal_resid_series = sub.loc[sub.Country == "Nepal", "resid"]
    nepal_resid = nepal_resid_series.iloc[0]
    pct = stats.percentileofscore(sub["resid"], nepal_resid)
    print(f"{col:28s} r={r:.3f} p={p:.4f} n={len(sub):3d} | Nepal residual={nepal_resid:+.2f} (percentile {pct:.1f})")

sub = test_df.dropna(subset=["Government digital policy"]).copy()
slope, intercept, r, p, se = stats.linregress(sub["log_gdp"], sub["Government digital policy"])
sub["resid"] = sub["Government digital policy"] - (slope * sub["log_gdp"] + intercept)
sub_sorted = sub.sort_values("resid")

nepal_rank = (sub_sorted.reset_index(drop=True).Country == "Nepal").idxmax() + 1
print(f"\nNepal's rank among worst residuals: {nepal_rank} of {len(sub_sorted)}")
print("\n--- Worst 15 income-adjusted Government Digital Policy residuals, full sample ---")
print(sub_sorted[["Country", "gdp_pc_ppp", "Government digital policy", "resid"]].head(15).to_string(index=False))

sub_sorted.to_csv(PROCESSED_DIR / "gov_digital_policy_income_adjusted_full_sample.csv", index=False)

Full-sample test, n=188 countries

Government digital policy    r=0.593 p=0.0000 n=188 | Nepal residual=-18.43 (percentile 22.3)
Compute capacity             r=0.494 p=0.0000 n=188 | Nepal residual=+3.10 (percentile 76.1)
Total Score                  r=0.731 p=0.0000 n=188 | Nepal residual=+8.07 (percentile 71.8)
Governance                   r=0.657 p=0.0000 n=188 | Nepal residual=+19.23 (percentile 84.6)
AI Infrastructure            r=0.810 p=0.0000 n=188 | Nepal residual=+4.06 (percentile 71.8)

Nepal's rank among worst residuals: 42 of 188

--- Worst 15 income-adjusted Government Digital Policy residuals, full sample ---
                         Country   gdp_pc_ppp  Government digital policy      resid
                          Guyana 97899.011618                      10.00 -53.783469
                         Andorra 79566.792587                      10.54 -50.182456
                      Seychelles 35854.028162                       0.87 -48.084043
                         Belarus

In [6]:
SOUTH_ASIA = ["Nepal", "India", "Pakistan", "Bangladesh", "Sri Lanka", "Bhutan", "Afghanistan", "Maldives"]
print(sub_sorted[sub_sorted.Country.isin(SOUTH_ASIA)][["Country", "gdp_pc_ppp", "Government digital policy", "resid"]].to_string(index=False))

    Country   gdp_pc_ppp  Government digital policy      resid
   Maldives 28554.894567                       2.12 -43.473513
      Nepal  6175.158972                       4.56 -18.426596
  Sri Lanka 17065.029864                      24.25 -13.743421
   Pakistan  6573.416134                      13.92  -9.989287
Afghanistan  2236.199073                       0.00  -7.990691
     Bhutan 19594.002838                      65.00  24.966409
      India 11747.916044                      71.19  38.708530
 Bangladesh 10153.708379                      72.12  41.791560
